In [6]:
import tensorflow as tf

In [7]:

# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms


# ==========================================
# 2. DEVICE CONFIGURATION
# ==========================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ==========================================
# 3. LOAD MNIST DATASET
# ==========================================

transform = transforms.ToTensor()

train_data = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_data = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)


# ==========================================
# 4. CREATE DATALOADERS
# ==========================================

train_loader = torch.utils.data.DataLoader(
    dataset=train_data,
    batch_size=64,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    dataset=test_data,
    batch_size=64,
    shuffle=False
)


# ==========================================
# 5. BUILD CNN MODEL
# ==========================================

class CNN(nn.Module):

    def __init__(self):
        super(CNN, self).__init__()

        self.features = nn.Sequential(

            # First Conv2D
            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=3,
                padding=0
            ),
            nn.ReLU(),

            # Second Conv2D
            nn.Conv2d(
                in_channels=32,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            # Third Conv2D
            nn.Conv2d(
                in_channels=32,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(

            # Flatten
            nn.Flatten(),

            # Dense Layer
            nn.Linear(26 * 26 * 32, 128),
            nn.ReLU(),

            # Output Layer
            nn.Linear(128, 10)
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x


# ==========================================
# 6. INITIALIZE MODEL
# ==========================================

model = CNN().to(device)

print(model)


# ==========================================
# 7. LOSS FUNCTION AND OPTIMIZER
# ==========================================

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# ==========================================
# 8. TRAIN MODEL
# ==========================================

epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward Pass
        predictions = model(images)

        # Calculate Loss
        loss = loss_fn(predictions, labels)

        # Clear Previous Gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update Weights
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {average_loss:.4f}"
    )


# ==========================================
# 9. EVALUATE MODEL
# ==========================================

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Predictions
        outputs = model(images)

        # Get Predicted Classes
        predicted = torch.argmax(outputs, dim=1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()


accuracy = 100 * correct / total

print(f"\nTest Accuracy: {accuracy:.2f}%")


# ==========================================
# 10. MAKE A SAMPLE PREDICTION
# ==========================================

model.eval()

images, labels = next(iter(test_loader))

sample_image = images[0].unsqueeze(0).to(device)

with torch.no_grad():

    output = model(sample_image)

    probabilities = torch.softmax(output, dim=1)

    predicted_class = torch.argmax(probabilities, dim=1)

print("\nActual Digit:", labels[0].item())

print("Predicted Digit:", predicted_class.item())

print(
    "Prediction Confidence:",
    f"{probabilities.max().item() * 100:.2f}%"
)


Using device: cpu


100.0%
100.0%
100.0%
100.0%


CNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU()
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=21632, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


KeyboardInterrupt: 